In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col, count, sum as _sum, when, row_number, desc, to_timestamp
import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import missingno as msno
import math
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.miscmodels.ordinal_model import OrderedModel
import matplotlib.patches as mpatches

In [2]:
spark = SparkSession.builder.appName("TER").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/01 12:22:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.option("delimiter", "\t").csv(
    '/kaggle/input/open-food-facts/en.openfoodfacts.org.products.csv',
    header=True
)

In [4]:
# Calculer le nombre total de lignes
total_rows = df.count()

In [ ]:
from pyspark.sql import functions as F
total_cells = total_rows * len(df.columns)

missing_cells = df.select([
    F.sum(F.col(c).isNull().cast("int")) for c in df.columns
]).toPandas().sum().sum()

print(f"Données manquantes : {missing_cells / total_cells * 100:.2f}%")

In [ ]:
threshold = 0.75

# Calculer le ratio de nulls pour chaque colonne
null_ratios = df.select([
    (_sum(when(
        col(c).isNull() | (col(c) == "") | (col(c) == "null"), 
        1
    ).otherwise(0)) / total_rows).alias(c) 
    for c in df.columns
]).first()

# Colonnes à garder (< 75% de nulls)
columns_to_keep = [c for c in df.columns if null_ratios[c] < threshold]

df_cleaned = df.select(columns_to_keep)

print(f"Conservées: {len(columns_to_keep)}/{len(df.columns)}")

In [ ]:
window = Window.partitionBy("code").orderBy(
    desc(to_timestamp("last_modified_datetime"))
)

df_cleaned = df_cleaned.withColumn("row_num", row_number().over(window)) \
                       .filter(col("row_num") == 1) \
                       .drop("row_num")

In [ ]:
window = Window.partitionBy("product_name", "brands").orderBy(desc(to_timestamp("last_modified_datetime")))

df_cleaned = (df_cleaned
    .withColumn("row_num", row_number().over(window))
    .filter(
        (col("row_num") == 1) |
        (col("product_name").isNull() & col("brands").isNull())
    )
    .drop("row_num")
)

In [ ]:
df_cleaned = df_cleaned.drop("url", "creator", "created_t", "last_modified_t", "last_modified_by", "last_updated_t", "last_updated_datetime", "brands", "brands_tags", "categories", "categories_tags", 'labels', 'labels_tags', 'countries', 'countries_tags', 'labels', 'labels_tags', 'countries', 'countries_tags:', 'food_groups', 'food_groups_tags', 'states', 'states_tags', 'last_image_t', 'last_image_datetime', 'main_category', 'image_url', 'image_small_url', 'image_ingredients_url', 'image_ingredients_small_url', 'image_nutrition_url', 'image_nutrition_small_url', 'nutrition-score-fr_100g')

In [ ]:
output_dir = "/kaggle/working/output_temp"

df_cleaned.coalesce(1).write \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("quote", "") \
    .option("escape", "\\") \
    .mode("overwrite") \
    .csv(output_dir)

csv_file = [f for f in os.listdir(output_dir) if f.startswith("part-")][0]
shutil.move(f"{output_dir}/{csv_file}", "/kaggle/working/df.csv")
shutil.rmtree(output_dir)

In [ ]:
df = pd.read_csv("/kaggle/working/df.csv", sep='\t', on_bad_lines='skip')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

def format_cell(x):
    if abs(x) >= 10000:
        return f"{x:.2e}"
    return f"{x:.2f}"

cols_100g = [col for col in df.columns if col.endswith('_100g')]

desc = (df[cols_100g].describe()
          .rename(columns={'fruits-vegetables-nuts-estimate-from-ingredients_100g': 'fvn_estimate_100g'}))

table = ax.table(
    cellText=desc.applymap(format_cell).values,
    colLabels=desc.columns,
    rowLabels=desc.index,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.auto_set_column_width(col=list(range(len(desc.columns))))
plt.tight_layout()
plt.savefig("describe.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['_nb_neg'] = (df[cols_100g] < 0).sum(axis=1)

max_neg = df['_nb_neg'].max()

from IPython.display import display

for n in range(1, max_neg + 1):
    sub = df[df['_nb_neg'] == n][cols_100g].head(5)
    if len(sub) > 0:
        print(f"Nombre de lignes avec {n} valeurs négatives : {len(df[df['_nb_neg'] == n][cols_100g])}")
        display(sub)
        print()

In [ ]:
sub_1neg = df[df['_nb_neg'] == 1]
neg_counts = {col: (sub_1neg[col] < 0).sum() for col in cols_100g if (sub_1neg[col] < 0).any()}
for col, count in sorted(neg_counts.items(), key=lambda x: -x[1]):
    print(f"{col} : {count}")


In [ ]:
df[df['energy-kcal_100g'] < 0][['energy-kcal_100g', 'energy_100g']]

In [ ]:
for col in cols_100g:
    df[col] = df[col].where(df[col] >= 0, other=np.nan)

print(f"Valeurs négatives remplacées par NaN dans {len(cols_100g)} colonnes numériques.")


In [ ]:
mask_fat = df['saturated-fat_100g'] > df['fat_100g']
df.loc[mask_fat, ['saturated-fat_100g', 'fat_100g']] = np.nan

mask_salt = df['sodium_100g'] > df['salt_100g']
df.loc[mask_salt, ['sodium_100g', 'salt_100g']] = np.nan

print(f"Valeurs corrigées (graisses) : {mask_fat.sum():,}")
print(f"Valeurs corrigées (sel/sodium) : {mask_salt.sum():}")

In [ ]:
df[df['energy-kcal_100g'] > 900]['energy-kcal_100g'].info()

In [ ]:
mask_kcal = df['energy-kcal_100g'] > 900
df.loc[mask_kcal, 'energy-kcal_100g'] = np.nan

mask_kj = df['energy_100g'] > 3700
df.loc[mask_kj, 'energy_100g'] = np.nan

print(f"Valeurs corrigées (kcal) : {mask_kcal.sum():,}")
print(f"Valeurs corrigées (kj) : {mask_kj.sum():,}")
print(f"Lignes restantes : {len(df):,}")

In [ ]:
cols_0_100 = [col for col in cols_100g if col not in ('energy-kcal_100g', 'energy_100g', 'nutrition-score-fr_100g')]

for col in cols_0_100:
    df[col] = df[col].where(df[col] <= 100, other=np.nan)

In [ ]:
df = df.drop(columns=['_nb_neg'])
df.describe()

In [ ]:
df.to_csv("/kaggle/working/df_cleaned.csv", sep="\t", index=False)

In [ ]:
msno.bar(df)

In [ ]:
msno.matrix(df)
plt.savefig("2.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
missing_pct = df.isnull().sum().sum() / df.size * 100
print(f"Données manquantes : {missing_pct:.2f}%")

In [ ]:
num_cols = df.select_dtypes(include='number').columns
fig, axes = plt.subplots(math.ceil(len(num_cols)/3), 3,
                         figsize=(18, math.ceil(len(num_cols)/3)*3),
                         constrained_layout=True)
for ax, col in zip(axes.flatten(), num_cols):
    df[col].dropna().hist(ax=ax, bins=40, color='steelblue')
    ax.set_title(col, fontsize=7)
[ax.set_visible(False) for ax in axes.flatten()[len(num_cols):]]
plt.show()

In [ ]:
cols = ["nutriscore_score", "nova_group"]
df_clean_new = df[cols].dropna().copy()

df_clean_new["nova_group"] = df_clean_new["nova_group"].astype(int)
df_clean_new = df_clean_new[df_clean_new["nova_group"].isin([1, 2, 3, 4])]

print(f"Lignes utilisées : {len(df_clean_new):,}")
print(f"\nDistribution des groupes NOVA :")
print(df_clean_new["nova_group"].value_counts().sort_index())

In [ ]:
# Histogramme global
plt.hist(
    df_clean_new["nutriscore_score"],
    bins=40, color="#3498db", edgecolor="white", alpha=0.85
)
plt.title("Distribution du Nutrition-score FR", fontsize=15, fontweight="bold")
plt.axvline(df_clean_new["nutriscore_score"].mean(), color="red", linestyle="--", label=f"Moyenne : {df_clean_new['nutriscore_score'].mean():.1f}")
plt.axvline(df_clean_new["nutriscore_score"].median(), color="orange", linestyle="--", label=f"Médiane : {df_clean_new['nutriscore_score'].median():.1f}")
plt.xlabel("Nutrition-score")
plt.ylabel("Fréquence")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
print("Statistiques descriptives du nutrition-score par groupe NOVA")
desc = df_clean_new.groupby("nova_group")["nutriscore_score"].agg(
    ["count", "mean", "median", "std", "min", "max"]
).round(2)
print(desc)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Nutrition-score selon le groupe NOVA", fontsize=15, fontweight="bold")

palette = {1: "#2ecc71", 2: "#f1c40f", 3: "#e67e22", 4: "#e74c3c"}

# Boxplot
sns.boxplot(
    data=df_clean_new, x="nova_group", y="nutriscore_score",
    hue="nova_group", palette=palette, legend=False, ax=axes[0]
)
axes[0].set_title("Boxplot")
axes[0].set_xlabel("Groupe NOVA (1=non transformé → 4=ultra-transformé)")
axes[0].set_ylabel("Nutrition-score FR (100g)")

# Violin plot
sns.violinplot(
    data=df_clean_new, x="nova_group", y="nutriscore_score",
    hue="nova_group", palette=palette, inner="quartile", legend=False, ax=axes[1]
)
axes[1].set_title("Violin plot")
axes[1].set_xlabel("Groupe NOVA (1=non transformé → 4=ultra-transformé)")
axes[1].set_ylabel("Nutrition-score FR (100g)")

plt.tight_layout()
plt.show()

In [ ]:
print("ANOVA")
groups = [
    df_clean_new[df_clean_new["nova_group"] == g]["nutriscore_score"].values
    for g in [1, 2, 3, 4]
]
f_stat, p_value = stats.f_oneway(*groups)
print(f"F-statistic : {f_stat:.2f}")
print(f"p-value     : {p_value:.4e}")
if p_value < 0.05:
    print("→ Différence significative entre les groupes NOVA (p < 0.05)")
else:
    print("→ Pas de différence significative détectée")

# Eta-squared
grand_mean = df_clean_new["nutriscore_score"].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = sum((df_clean_new["nutriscore_score"] - grand_mean) ** 2)
eta_squared = ss_between / ss_total
print(f"Eta² (taille d'effet) : {eta_squared:.3f}")
print("  Interprétation : 0.01=petit, 0.06=moyen, 0.14=grand")

In [ ]:
print("Test post-hoc de Tukey (comparaisons par paires)")
tukey = pairwise_tukeyhsd(
    endog=df_clean_new["nutriscore_score"],
    groups=df_clean_new["nova_group"],
    alpha=0.05
)
print(tukey)

In [ ]:
print("Corrélation de Spearman")
rho, p_spearman = stats.spearmanr(
    df_clean_new["nova_group"], df_clean_new["nutriscore_score"]
)
print(f"ρ de Spearman : {rho:.3f}")
print(f"p-value       : {p_spearman:.4e}")
print("Interprétation : ρ > 0 → plus le groupe NOVA est élevé, plus le score est mauvais")

In [ ]:
print("Régression ordinale")
sample = df_clean_new.sample(min(len(df_clean_new), 20_000), random_state=42).copy()
sample["nova_group"] = sample["nova_group"].astype("category")

model = OrderedModel(
    endog=sample["nova_group"],
    exog=sample[["nutriscore_score"]],
    distr="logit"
)
result = model.fit(method="bfgs", disp=False)
print(result.summary())

print("Odds Ratios (régression ordinale logit)")
odds_ratios = np.exp(result.params)
print(odds_ratios)
print("\nInterprétation : OR > 1 → une augmentation du score augmente la probabilité d'appartenir à un groupe NOVA plus élevé (ultra-transformé)")

In [ ]:
df_nutri = df[cols_100g + ['nutriscore_score', 'nutriscore_grade', 'product_name', 'categories_en']]

missing_counts = df_nutri.isnull().sum()
missing_pct = (missing_counts / len(df_nutri) * 100).round(2)

missing_df = pd.DataFrame({
    'Colonne': missing_counts.index,
    'Valeurs manquantes': missing_counts.values,
    'Pourcentage (%)': missing_pct.values
}).sort_values('Pourcentage (%)', ascending=False)

print(missing_df.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = [
    '#e74c3c' if p > 50 else '#e67e22' if p > 20 else '#27ae60'
    for p in missing_df['Pourcentage (%)']
]

bars = ax.barh(missing_df['Colonne'], missing_df['Pourcentage (%)'], color=colors)
ax.set_xlabel("Pourcentage de donnees manquantes (%)", fontsize=12)
ax.set_title("Taux de donnees manquantes - Variables Nutri-Score", fontsize=14, fontweight='bold')
ax.axvline(x=20, color='orange', linestyle='--', alpha=0.7, label='Seuil 20%')
ax.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='Seuil 50%')
ax.legend()

for bar, pct in zip(bars, missing_df['Pourcentage (%)']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig("Taux de donnees manquantes - Variables Nutri-Score.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
msno.matrix(df_nutri, ax=ax, sparkline=False, fontsize=10)
ax.set_title(f"Matrice des donnees manquantes",
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
msno.heatmap(df_nutri, ax=ax, fontsize=9)
ax.set_title("Correlations des donnees manquantes - Variables Nutri-Score",
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("Correlations des donnees manquantes - Variables Nutri-Score.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
msno.dendrogram(df_nutri, ax=ax, fontsize=10)
ax.set_title("Dendrogramme des donnees manquantes", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("Dendrogramme des donnees manquantes.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if 'pnns_groups_1' in df.columns:
    df_pnns = df[['pnns_groups_1'] + cols_100g].copy()
    df_pnns = df_pnns[
        df_pnns['pnns_groups_1'].notna() &
        (df_pnns['pnns_groups_1'] != 'unknown')
    ]

    completeness = (
        df_pnns.groupby('pnns_groups_1')[cols_100g]
        .apply(lambda x: x.notna().mean().mean() * 100)
        .sort_values(ascending=True)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    completeness.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel("Taux de completude moyen (%)", fontsize=11)
    ax.set_title("Completude des donnees Nutri-Score par groupe PNNS1",
                 fontsize=13, fontweight='bold')
    ax.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='50% completude')
    ax.legend()
    for i, v in enumerate(completeness.values):
        ax.text(v + 0.5, i, f"{v:.1f}%", va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig("Completude des donnees Nutri-Score par groupe PNNS1.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Colonne pnns_groups_1 non disponible.")


In [ ]:
if 'pnns_groups_2' in df.columns:
    df_pnns = df[['pnns_groups_2'] + cols_100g].copy()
    df_pnns = df_pnns[
        df_pnns['pnns_groups_2'].notna() &
        (df_pnns['pnns_groups_2'] != 'unknown')
    ]

    completeness = (
        df_pnns.groupby('pnns_groups_2')[cols_100g]
        .apply(lambda x: x.notna().mean().mean() * 100)
        .sort_values(ascending=True)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    completeness.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel("Taux de completude moyen (%)", fontsize=11)
    ax.set_title("Completude des donnees Nutri-Score par groupe PNNS2",
                 fontsize=13, fontweight='bold')
    ax.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='50% completude')
    ax.legend()
    for i, v in enumerate(completeness.values):
        ax.text(v + 0.5, i, f"{v:.1f}%", va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig("Completude des donnees Nutri-Score par groupe PNNS2.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Colonne pnns_groups_1 non disponible.")


In [ ]:
cols_nutriscore = [
    'product_name',
    'energy_100g',                                          # e : énergie (kJ)
    'saturated-fat_100g',                                   # g : graisses saturées
    'sugars_100g',                                          # s : sucres
    'sodium_100g',                                          # n : sodium
    'fruits-vegetables-nuts-estimate-from-ingredients_100g',# f : fruits/légumes (%)
    'fiber_100g',                                           # phi : fibres
    'proteins_100g'                                         # p : protéines
]

df_nutri = df[cols_nutriscore].copy()

df_nutri.columns = ['nom', 'energie', 'graisses', 'sucres', 'sodium', 
                    'fruits', 'fibres', 'proteines']
df_nutri = df_nutri.dropna()
print(df_nutri.shape)

In [ ]:
def Ene(e):
    if e <= 335:    return 0
    elif e <= 670:  return 1
    elif e <= 1005: return 2
    elif e <= 1340: return 3
    elif e <= 1675: return 4
    elif e <= 2010: return 5
    elif e <= 2345: return 6
    elif e <= 2680: return 7
    elif e <= 3015: return 8
    elif e <= 3350: return 9
    else:           return 10

def Gra(g):
    if g <= 1:  return 0
    elif g <= 2:  return 1
    elif g <= 3:  return 2
    elif g <= 4:  return 3
    elif g <= 5:  return 4
    elif g <= 6:  return 5
    elif g <= 7:  return 6
    elif g <= 8:  return 7
    elif g <= 9:  return 8
    elif g <= 10: return 9
    else:         return 10

def Suc(s):
    if s <= 4.5:    return 0
    elif s <= 9:    return 1
    elif s <= 13.5: return 2
    elif s <= 18:   return 3
    elif s <= 22.5: return 4
    elif s <= 27:   return 5
    elif s <= 31:   return 6
    elif s <= 36:   return 7
    elif s <= 40:   return 8
    elif s <= 45:   return 9
    else:           return 10

def Sod(n):
    n = n * 1000  # conversion g -> mg
    if n <= 90:    return 0
    elif n <= 180: return 1
    elif n <= 270: return 2
    elif n <= 360: return 3
    elif n <= 450: return 4
    elif n <= 540: return 5
    elif n <= 630: return 6
    elif n <= 720: return 7
    elif n <= 810: return 8
    elif n <= 900: return 9
    else:          return 10

def A(row):
    return Ene(row['energie']) + Gra(row['graisses']) + Suc(row['sucres']) + Sod(row['sodium'])

df_nutri['A'] = df_nutri.apply(A, axis=1)

In [ ]:
def Fru(f):
    if f <= 40:   return 0
    elif f <= 60: return 1
    elif f <= 80: return 2
    else:         return 5

def Fib(phi):
    if phi <= 0.9:  return 0
    elif phi <= 1.9: return 1
    elif phi <= 2.8: return 2
    elif phi <= 3.7: return 3
    elif phi <= 4.7: return 4
    else:            return 5

def Pro(p):
    if p <= 1.6:  return 0
    elif p <= 3.2: return 1
    elif p <= 4.8: return 2
    elif p <= 6.4: return 3
    elif p <= 8.0: return 4
    else:          return 5

def C(row):
    return Fru(row['fruits']) + Fib(row['fibres']) + Pro(row['proteines'])

df_nutri['C'] = df_nutri.apply(C, axis=1)

In [ ]:
def Score(row):
    a = row['A']
    c = row['C']
    fru = Fru(row['fruits'])
    fib = Fib(row['fibres'])
    
    if a < 11:
        return a - c
    elif fru == 5:
        return a - c
    else:
        return a - (fru + fib)

def Let(score):
    if score <= -1:   return 'A'
    elif score <= 2:  return 'B'
    elif score <= 10: return 'C'
    elif score <= 18: return 'D'
    else:             return 'E'

df_nutri['score'] = df_nutri.apply(Score, axis=1)
df_nutri['lettre'] = df_nutri['score'].apply(Let)

In [ ]:
sample = df_nutri.sample(1000, random_state=42)

couleurs_map = {'A': '#2d8b4e', 'B': '#85bb2f', 'C': '#f5a623', 'D': '#e8604c', 'E': '#cc1a1a'}
zones = [(-15, -1, '#2d8b4e', 'A'), (-1, 2, '#85bb2f', 'B'), 
         (2, 10, '#f5a623', 'C'), (10, 18, '#e8604c', 'D'), (18, 45, '#cc1a1a', 'E')]

fig, ax = plt.subplots(figsize=(14, 8))

for y_min, y_max, couleur, lettre in zones:
    ax.axhspan(y_min, y_max, alpha=0.08, color=couleur)

for lettre, groupe in sample.groupby('lettre'):
    ax.scatter(groupe['A'], groupe['C'],
               c=couleurs_map[lettre], alpha=0.6, s=30, zorder=3)

for lettre in ['A', 'B', 'C', 'D', 'E']:
    sous = sample[sample['lettre'] == lettre].dropna(subset=['nom'])
    if len(sous) > 0:
        row = sous.iloc[0]
        nom = str(row['nom'])[:25]
        ax.annotate(nom,
                    xy=(row['A'], row['C']),
                    xytext=(row['A'] + 0.3, row['C'] + 0.3),
                    fontsize=7,
                    color=couleurs_map[lettre],
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

# Mise en forme
ax.set_xlabel('Points A (énergie, graisses, sucres, sodium)', fontsize=11)
ax.set_ylabel('PointsC (fruits, fibres, protéines)', fontsize=11)
ax.set_title('Nutri-Score calculé sur une échantillon de 1000 produits',
             fontsize=12, pad=15)

# Légende
patches = [mpatches.Patch(color=c, label=f'Nutri-Score {l}') 
           for l, c in couleurs_map.items()]
ax.legend(handles=patches, loc='upper right', framealpha=0.9, fontsize=9)

ax.set_xlim(-1, 42)
ax.set_ylim(-1, 16)
plt.savefig('nutriscore_plot.png', dpi=150, bbox_inches='tight')
plt.show()